#### THIRD ATTEMPT

## Data Modelling

### OpTc Dataset
Overview
The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC&utm_source=chatgpt.com



**Project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.



**Chapter goal:**

The goal of this chapter is to use the cleaned OpTC telemetry generated in the previous chapter to train and evaluate ML/DL models for malicious event detection. The models will attempt to classify individual telemetry events as benign or malicious. This also allows me to apply the modelling pipeline used for the previous datasets to the more complex OpTC telemetry and compare model performance before moving on to the main attack prediction experiment.

In [22]:
# Imports
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
import tensorflow as tf

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Libraries for models
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Libraries for evaluation
from sklearn import metrics

# Libraries for deep learning
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.layers import Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from pathlib import Path
import pyarrow.parquet as pq


# Ignore warnings
import warnings
warnings.filterwarnings("ignore")





In [4]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Load and preview the data

In [5]:
# Location of ready flattened parquet files in Google Drive
data_path = Path("/content/drive/MyDrive/solutions/OpTC_sample_cleaned")

# Get the parquet files
files = list(data_path.glob("*.parquet"))
print(f"Number of files: {len(files)}")



# Check the number of rows in each file
for file in files:

    data = pd.read_parquet(
        file,
        columns=["hostname", "label"]
    )

    print(
        file.name,
        "| Rows:", f"{len(data):,}",
        "| Malicious:", f"{data['label'].sum():,}"
    )


Number of files: 12
2019-09-24_AIA-501-525_sysclient0502.parquet | Rows: 4,246,693 | Malicious: 0
2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet | Rows: 136,241 | Malicious: 0
2019-09-25_AIA-351-375_sysclient0352.parquet | Rows: 2,476,116 | Malicious: 0
2019-09-24_AIA-801-825_sysclient0811.parquet | Rows: 3,818,669 | Malicious: 18,649
2019-09-25_AIA-51-75_sysclient0052.parquet | Rows: 2,406,290 | Malicious: 0
2019-09-23_AIA-201-225_sysclient0202.parquet | Rows: 4,706,585 | Malicious: 0
2019-09-25_AIA-351-375_sysclient0351.parquet | Rows: 2,518,452 | Malicious: 18,889
2019-09-24_AIA-801-825_sysclient0812.parquet | Rows: 3,787,461 | Malicious: 0
2019-09-24_AIA-501-525_sysclient0501.parquet | Rows: 4,383,849 | Malicious: 27,577
2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0052.parquet | Rows: 146,509 | Malicious: 0
2019-09-23_AIA-201-225_sysclient0201.parquet | Rows: 4,376,026 | Malicious: 26,681
2019-09-25_AIA-51-75_sysclient0051.parquet | Rows: 2,524,447

### 2. Further Data Inspection

In [6]:
# Load one cleaned file to inspect the available features
data = pd.read_parquet(files[0])

print("Columns:")
print(data.columns.tolist())

print("\nData types:")
print(data.dtypes)

Columns:
['action', 'hostname', 'object', 'pid', 'ppid', 'principal', 'tid', 'timestamp', 'label', 'acuity_level', 'base_address', 'command_line', 'dest_ip', 'dest_port', 'direction', 'file_path', 'image_path', 'info_class', 'l4protocol', 'module_path', 'parent_image_path', 'size', 'src_ip', 'src_pid', 'src_port', 'src_tid', 'stack_base', 'stack_limit', 'start_address', 'subprocess_tag', 'tgt_pid', 'tgt_tid', 'user_stack_base', 'user_stack_limit']

Data types:
action                            object
hostname                          object
object                            object
pid                                int64
ppid                               int64
principal                         object
tid                                int64
timestamp            datetime64[ns, UTC]
label                              int64
acuity_level                      object
base_address                      object
command_line                      object
dest_ip                           object
de

In [7]:
# Identify numerical columns
numeric_cols = data.select_dtypes(
    include=np.number
).columns

print("Numerical columns:")
print(numeric_cols.tolist())

Numerical columns:
['pid', 'ppid', 'tid', 'label']


In [8]:
# Identify categorical columns
categorical_cols = data.select_dtypes(
    include=["object", "string"]
).columns

# Get the number of unique values of categorical variables
# so as to know if encoding would make data explode

print("Categorical columns and unique values:")

for col in categorical_cols:
    print(f"{col}: {data[col].nunique():,}")

Categorical columns and unique values:
action: 22
hostname: 1
object: 9
principal: 30
acuity_level: 6
base_address: 5,221
command_line: 376
dest_ip: 78
dest_port: 6,888
direction: 2
file_path: 15,956
image_path: 124
info_class: 7
l4protocol: 5
module_path: 2,237
parent_image_path: 116
size: 30,081
src_ip: 6,880
src_pid: 1,348
src_port: 16,417
src_tid: 1,514
stack_base: 11,491
stack_limit: 11,491
start_address: 1,193
subprocess_tag: 94
tgt_pid: 1,348
tgt_tid: 1,503
user_stack_base: 59,952
user_stack_limit: 65,823


### 2. Initital feature Selection

In [9]:
# Select features for the initial detection experiment
# I am doing this to save RAM actually
# I had to include hostname, but will remove it later after the split

selected_features = [
    "action",
    "object",
    "acuity_level",
    "direction",
    "info_class",
    "l4protocol",
    "pid",
    "ppid",
    "tid",
    "timestamp",
    "label",
    "hostname"
]



### 3. Creating a DataFrame

In [10]:
# Load only the selected features from each file
# and use this to create a dataframe


data_parts = []

for file in files:

    part = pd.read_parquet(
        file,
        columns=selected_features
    )

    data_parts.append(part)


# Combine the selected data
data = pd.concat(
    data_parts,
    ignore_index=True
)

del data_parts

print(f"Dataset shape: {data.shape}")

print("\nLabel distribution:")
print(data["label"].value_counts())

print("\nLabel percentages:")
print(data["label"].value_counts(normalize=True) * 100)

Dataset shape: (35527338, 12)

Label distribution:
label
0    35430397
1       96941
Name: count, dtype: int64

Label percentages:
label
0    99.727137
1     0.272863
Name: proportion, dtype: float64


In [11]:
# # Check the number of benign and malicious events for each host
# host_distribution = pd.crosstab(
#     data["hostname"],
#     data["label"]
# )

# host_distribution.columns = ["benign", "malicious"]

# print(host_distribution)

# # Check the dates available for each host
# data["date"] = data["timestamp"].dt.date

# print(
#     data.groupby("hostname")["date"]
#     .unique()
# )

### 4. Data Splitting (Host and time conscious)

In [12]:
# Specify the hosts to keep for testing

test_hosts = [
    "sysclient0351",
    "sysclient0352"
]

# Separate the files into training and testing files
train_files = [
    file for file in files
    if not any(host in file.name.lower() for host in test_hosts)
]

test_files = [
    file for file in files
    if any(host in file.name.lower() for host in test_hosts)
]

print(f"Training files: {len(train_files)}")
print(f"Testing files: {len(test_files)}")

Training files: 10
Testing files: 2


### 5. Separate maliciouss and benig parts

In [13]:
# Features needed for the detection experiment
model_features = [
    "action",
    "object",
    "acuity_level",
    "direction",
    "info_class",
    "l4protocol",
    "pid",
    "ppid",
    "tid",
    "timestamp",
    "label"
]


# Store malicious and benign events separately
malicious_parts = []
benign_parts = []

for file in train_files:

    print(f"Processing: {file.name}")

    part = pd.read_parquet(
        file,
        columns=model_features
    )

    # Keep all malicious events
    malicious_part = part[
        part["label"] == 1
    ]

    if len(malicious_part) > 0:
        malicious_parts.append(malicious_part)

    # Store benign events temporarily
    benign_part = part[
        part["label"] == 0
    ]

    benign_parts.append(benign_part)

    del part

Processing: 2019-09-24_AIA-501-525_sysclient0502.parquet
Processing: 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet
Processing: 2019-09-24_AIA-801-825_sysclient0811.parquet
Processing: 2019-09-25_AIA-51-75_sysclient0052.parquet
Processing: 2019-09-23_AIA-201-225_sysclient0202.parquet
Processing: 2019-09-24_AIA-801-825_sysclient0812.parquet
Processing: 2019-09-24_AIA-501-525_sysclient0501.parquet
Processing: 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0052.parquet
Processing: 2019-09-23_AIA-201-225_sysclient0201.parquet
Processing: 2019-09-25_AIA-51-75_sysclient0051.parquet


### 6. Create Split Samples

In [14]:
# Create a 1:10 malicious-to-benign training sample

total_malicious = sum(
    pd.read_parquet(file, columns=["label"])["label"].sum()
    for file in train_files
)

train_parts = []

for file in train_files:
    part = pd.read_parquet(file, columns=selected_features)

    malicious = part[part["label"] == 1]

    # Sample a share of benign events from every host
    benign = part[part["label"] == 0].sample(
        n=min(len(part[part["label"] == 0]), total_malicious),
        random_state=7
    )

    train_parts.append(pd.concat([malicious, benign]))

train_data = pd.concat(train_parts, ignore_index=True)

# Reduce benign events to the final 1:10 ratio
malicious = train_data[train_data["label"] == 1]
benign = train_data[train_data["label"] == 0].sample(
    n=len(malicious) * 10,
    random_state=7
)

train_data = pd.concat([malicious, benign], ignore_index=True).sample(
    frac=1, random_state=7
).reset_index(drop=True)

print(train_data["label"].value_counts())

label
0    780520
1     78052
Name: count, dtype: int64


In [15]:
# Load the held-out test hosts
test_data = pd.concat(
    [pd.read_parquet(file, columns=selected_features) for file in test_files],
    ignore_index=True
)

print(test_data["label"].value_counts())

# Extract time features
for dataset in [train_data, test_data]:
    dataset["hour"] = dataset["timestamp"].dt.hour
    dataset["minute"] = dataset["timestamp"].dt.minute

# Define independent and dependent variables
x_train = train_data.drop(columns=["label", "timestamp"])
y_train = train_data["label"]

x_test = test_data.drop(columns=["label", "timestamp"])
y_test = test_data["label"]

label
0    4975679
1      18889
Name: count, dtype: int64


### 7. Prepare train and test data

In [16]:
# Extract time features
for dataset in [train_data, test_data]:
    dataset["hour"] = dataset["timestamp"].dt.hour
    dataset["minute"] = dataset["timestamp"].dt.minute

# Remove timestamp and hostname, then separate features and labels
x_train = train_data.drop(columns=["label", "timestamp", "hostname"], errors="ignore")
y_train = train_data["label"]

x_test = test_data.drop(columns=["label", "timestamp", "hostname"], errors="ignore")
y_test = test_data["label"]

### 8. Encode Categorical Features

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify categorical and numerical columns
cat_cols = x_train.select_dtypes(include=["object"]).columns
numeric_cols = x_train.select_dtypes(include=np.number).columns

# Encode categorical features and scale numerical features
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), cat_cols),
    ("num", StandardScaler(), numeric_cols)
], sparse_threshold=1.0)

X = preprocessor.fit_transform(x_train).astype(np.float32)
X_TEST = preprocessor.transform(x_test).astype(np.float32)

Y = y_train.copy()
Y_TEST = y_test.copy()

print(X.shape)
print(X_TEST.shape)

(858572, 60)
(4994568, 60)


### 9. Define and Train ML models

In [27]:
# Define models
models = [
    ("Logistic Regression", LogisticRegression(
        max_iter=1000,
        random_state=7,
        class_weight="balanced"
    )),

    ("Bernoulli NB", BernoulliNB()),

    ("Decision Tree", DecisionTreeClassifier(
        random_state=7,
        class_weight="balanced",
        max_depth=10
    )),

    ("Random Forest", RandomForestClassifier(
        n_estimators=100,
        random_state=7,
        class_weight="balanced",
        n_jobs=-1
    )),

    ("XGBoost", XGBClassifier(
        random_state=7,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss",
        n_jobs=-1
    ))
]

In [29]:
# Train and evaluate the models on training data

for name, model in models:

    # Train model
    model.fit(X, Y)

    # Make predictions on training data
    predictions = model.predict(X)

    accuracy = metrics.accuracy_score(Y, predictions)
    conf_matrix = metrics.confusion_matrix(Y, predictions)
    report = metrics.classification_report(Y, predictions)

    print(f"\n===== {name} - Train Evaluation =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Logistic Regression - Train Evaluation =====
Accuracy: 0.9390907227349599
Confusion Matrix:
 [[731385  49135]
 [  3160  74892]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97    780520
           1       0.60      0.96      0.74     78052

    accuracy                           0.94    858572
   macro avg       0.80      0.95      0.85    858572
weighted avg       0.96      0.94      0.95    858572


===== Bernoulli NB - Train Evaluation =====
Accuracy: 0.916495063896796
Confusion Matrix:
 [[716853  63667]
 [  8028  70024]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.92      0.95    780520
           1       0.52      0.90      0.66     78052

    accuracy                           0.92    858572
   macro avg       0.76      0.91      0.81    858572
weighted avg       0.95      0.92      0.93    858572


===== Decision Tree - Train Evaluat

### 10. Validate on Test data

In [30]:
# Evaluate the models on test data

for name, model in models:

    predictions = model.predict(X_TEST)

    accuracy = metrics.accuracy_score(Y_TEST, predictions)
    conf_matrix = metrics.confusion_matrix(Y_TEST, predictions)
    report = metrics.classification_report(Y_TEST, predictions)

    print(f"\n===== {name} - Test Evaluation =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Logistic Regression - Test Evaluation =====
Accuracy: 0.9509677313433313
Confusion Matrix:
 [[4746245  229434]
 [  15461    3428]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.97   4975679
           1       0.01      0.18      0.03     18889

    accuracy                           0.95   4994568
   macro avg       0.51      0.57      0.50   4994568
weighted avg       0.99      0.95      0.97   4994568


===== Bernoulli NB - Test Evaluation =====
Accuracy: 0.9356348737268169
Confusion Matrix:
 [[4670615  305064]
 [  16412    2477]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97   4975679
           1       0.01      0.13      0.02     18889

    accuracy                           0.94   4994568
   macro avg       0.50      0.53      0.49   4994568
weighted avg       0.99      0.94      0.96   4994568


===== Decision Tree - Test E

### 11. DL Models - ANN

In [ ]:
# Convert sparse data for ANN
X_ann = X.toarray().astype("float32")
X_TEST_ann = X_TEST.toarray().astype("float32")


model = Sequential([

    Dense(
        256,
        activation='relu',
        kernel_regularizer=regularizers.l2(0.0005)
    ),
    BatchNormalization(),
    Dropout(0.4),

    Dense(
        128,
        activation='relu',
        kernel_regularizer=regularizers.l2(0.0005)
    ),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model.fit(
    X_ann, Y,
    epochs=20,
    batch_size=32,
    validation_data=(X_TEST_ann, Y_TEST),
    callbacks=[early_stop]
)


train_pred_probs = model.predict(X_ann)
train_pred_classes = (train_pred_probs > 0.5).astype(int).ravel()

print("Train Accuracy:", metrics.accuracy_score(Y, train_pred_classes))
print(metrics.classification_report(Y, train_pred_classes))


pred_probs = model.predict(X_TEST_ann)
pred_classes = (pred_probs > 0.5).astype(int).ravel()

print("Test Accuracy:", metrics.accuracy_score(Y_TEST, pred_classes))
print(metrics.classification_report(Y_TEST, pred_classes))

### 12. DL Models - CNN

In [ ]:
# Reshape input for Conv1D: (samples, features, channels)
X_cnn = np.expand_dims(X_ann, axis=2)
X_TEST_cnn = np.expand_dims(X_TEST_ann, axis=2)


# Build the CNN
cnn_classifier = Sequential()

cnn_classifier.add(
    Conv1D(
        64,
        kernel_size=3,
        activation='relu',
        padding='same',
        input_shape=(X_cnn.shape[1], 1)
    )
)

cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(
    Conv1D(
        128,
        kernel_size=3,
        activation='relu',
        padding='same'
    )
)

cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(
    Conv1D(
        256,
        kernel_size=3,
        activation='relu',
        padding='same'
    )
)

cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D())
cnn_classifier.add(Dropout(0.3))

cnn_classifier.add(Flatten())
cnn_classifier.add(Dense(64, activation='relu'))
cnn_classifier.add(Dropout(0.3))
cnn_classifier.add(Dense(1, activation='sigmoid'))

cnn_classifier.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop_cnn = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

cnn_history = cnn_classifier.fit(
    X_cnn, Y,
    validation_data=(X_TEST_cnn, Y_TEST),
    batch_size=16,
    epochs=20,
    callbacks=[early_stop_cnn]
)


train_pred_probs_cnn = cnn_classifier.predict(X_cnn)
train_pred_classes_cnn = (train_pred_probs_cnn > 0.5).astype(int).ravel()

print("Train Accuracy:", metrics.accuracy_score(Y, train_pred_classes_cnn))
print(metrics.classification_report(Y, train_pred_classes_cnn))


pred_probs_cnn = cnn_classifier.predict(X_TEST_cnn)
pred_classes_cnn = (pred_probs_cnn > 0.5).astype(int).ravel()

print("Test Accuracy:", metrics.accuracy_score(Y_TEST, pred_classes_cnn))
print(metrics.classification_report(Y_TEST, pred_classes_cnn))